# Segmentação RFM de Clientes

## Objetivo

Este notebook aplica a metodologia RFM para analisar o comportamento dos clientes da **Distribuidora Horizonte**.

RFM representa três dimensões:

- **Recência (R):** há quantos dias o cliente realizou sua última compra;
- **Frequência (F):** quantidade de pedidos realizados;
- **Valor Monetário (M):** faturamento acumulado do cliente.

A partir dessas métricas, os clientes recebem scores de 1 a 5 e são classificados em segmentos comportamentais.

## Segmentos

- Campeões;
- Leais;
- Potenciais leais;
- Novos clientes;
- Grandes clientes em risco;
- Em risco;
- Hibernando;
- Regulares;
- Sem compra.

## Importante

Os perfis utilizados internamente durante a geração dos dados fictícios não são utilizados nesta análise.

A classificação é realizada exclusivamente a partir das transações existentes na camada Silver.

## Saídas

Este notebook cria:

- `clientes_rfm`;
- `resumo_segmentos_rfm`.

## Fluxo

Silver → Métricas RFM → Scores → Segmentação → Gold

## Resultado

A segmentação RFM foi construída a partir exclusivamente do histórico transacional dos clientes.

Foram calculadas as métricas:

- recência;
- frequência;
- valor monetário;
- ticket médio;
- lucro acumulado;
- tempo de relacionamento.

Os clientes receberam scores de 1 a 5 para Recência, Frequência e Valor Monetário e foram classificados em grupos comportamentais.

A análise permite identificar diferentes estratégias de relacionamento, como:

- retenção de grandes clientes em risco;
- fidelização de clientes leais;
- desenvolvimento de novos clientes;
- recuperação de clientes hibernando;
- manutenção do relacionamento com clientes campeões.

Nenhum perfil definido durante a geração dos dados fictícios foi utilizado na classificação.

Os segmentos foram derivados exclusivamente do comportamento observado nas vendas.

In [0]:
from datetime import timedelta

from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"
schema_gold = "varejo_gold"


print(f"Catálogo: {catalogo_atual}")
print(f"Origem: {schema_silver}")
print(f"Destino: {schema_gold}")

In [0]:
def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


def salvar_gold(
    dataframe,
    nome_tabela
):
    """
    Salva um DataFrame como tabela Delta
    na camada Gold.
    """

    nome_completo = (
        f"{catalogo_atual}."
        f"{schema_gold}."
        f"{nome_tabela}"
    )

    (
        dataframe
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(
            nome_completo
        )
    )

    print(
        f"Tabela criada: {nome_completo}"
    )

In [0]:
df_resultado_qualidade = (

    spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"resultado_qualidade"
    )
)


testes_criticos_reprovados = (

    df_resultado_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        "Existem testes críticos de qualidade "
        "reprovados. A análise RFM não será processada."
    )


print(
    "Quality Gate aprovado."
)

In [0]:
df_clientes = carregar_silver(
    "dim_cliente"
)


df_vendas = carregar_silver(
    "fato_vendas"
)

In [0]:
ultima_data_venda = (

    df_vendas

    .agg(
        F.max(
            "data_venda"
        ).alias(
            "ultima_data_venda"
        )
    )

    .first()[
        "ultima_data_venda"
    ]
)


data_referencia = (
    ultima_data_venda
    + timedelta(days=1)
)


print(
    f"Última venda: {ultima_data_venda}"
)

print(
    f"Data de referência RFM: {data_referencia}"
)

In [0]:
df_metricas_rfm = (

    df_vendas

    .groupBy(
        "id_cliente"
    )

    .agg(

        F.min(
            "data_venda"
        ).alias(
            "data_primeira_compra"
        ),

        F.max(
            "data_venda"
        ).alias(
            "data_ultima_compra"
        ),

        F.countDistinct(
            "id_venda"
        ).alias(
            "frequencia"
        ),

        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "valor_monetario"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        )
    )

    .withColumn(

        "recencia_dias",

        F.datediff(
            F.lit(
                data_referencia
            ),
            F.col(
                "data_ultima_compra"
            )
        )
    )

    .withColumn(

        "ticket_medio",

        F.round(
            F.col(
                "valor_monetario"
            )
            /
            F.col(
                "frequencia"
            ),
            2
        )
    )

    .withColumn(

        "tempo_relacionamento_dias",

        F.datediff(
            F.lit(
                data_referencia
            ),
            F.col(
                "data_primeira_compra"
            )
        )
    )
)

In [0]:
display(
    df_metricas_rfm
    .orderBy(
        F.desc(
            "valor_monetario"
        )
    )
    .limit(20)
)

In [0]:
df_rfm_base = (

    df_clientes

    .select(
        "id_cliente",
        "nome_cliente",
        "segmento_cliente",
        "porte_cliente",
        "cidade",
        "estado",
        "situacao_cliente",
        "data_cadastro"
    )

    .join(
        df_metricas_rfm,
        on="id_cliente",
        how="left"
    )
)

In [0]:
clientes_sem_compra = (

    df_rfm_base

    .filter(
        F.col(
            "frequencia"
        ).isNull()
    )

    .count()
)


print(
    f"Clientes sem compra: "
    f"{clientes_sem_compra:,}"
)

In [0]:
df_clientes_com_compra = (

    df_rfm_base

    .filter(
        F.col(
            "frequencia"
        ).isNotNull()
    )
)

In [0]:
janela_recencia = (

    Window.orderBy(
        F.desc(
            "recencia_dias"
        )
    )
)


df_rfm_scores = (

    df_clientes_com_compra

    .withColumn(

        "score_r",

        F.ntile(5)
        .over(
            janela_recencia
        )
    )
)

In [0]:
janela_frequencia = (

    Window.orderBy(
        F.asc(
            "frequencia"
        )
    )
)


df_rfm_scores = (

    df_rfm_scores

    .withColumn(

        "score_f",

        F.ntile(5)
        .over(
            janela_frequencia
        )
    )
)

In [0]:
janela_monetario = (

    Window.orderBy(
        F.asc(
            "valor_monetario"
        )
    )
)


df_rfm_scores = (

    df_rfm_scores

    .withColumn(

        "score_m",

        F.ntile(5)
        .over(
            janela_monetario
        )
    )
)

In [0]:
df_rfm_scores = (

    df_rfm_scores

    .withColumn(

        "codigo_rfm",

        F.concat(
            F.col("score_r"),
            F.col("score_f"),
            F.col("score_m")
        )
    )
)

In [0]:
df_rfm_scores = (

    df_rfm_scores

    .withColumn(

        "score_rfm_medio",

        F.round(

            (
                F.col("score_r")
                +
                F.col("score_f")
                +
                F.col("score_m")
            )
            / 3,

            2
        )
    )
)

In [0]:
df_rfm_segmentado = (

    df_rfm_scores

    .withColumn(

        "segmento_rfm",

        # Clientes excelentes nas três dimensões

        F.when(

            (F.col("score_r") >= 4)
            &
            (F.col("score_f") >= 4)
            &
            (F.col("score_m") >= 4),

            "Campeões"
        )


        # Clientes frequentes e ainda recentes

        .when(

            (F.col("score_r") >= 3)
            &
            (F.col("score_f") >= 4),

            "Leais"
        )


        # Alta recência e frequência intermediária

        .when(

            (F.col("score_r") >= 4)
            &
            (F.col("score_f") == 3),

            "Potenciais leais"
        )


        # Compraram recentemente,
        # mas ainda possuem baixa frequência

        .when(

            (F.col("score_r") >= 4)
            &
            (F.col("score_f") <= 2),

            "Novos clientes"
        )


        # Clientes historicamente muito importantes
        # que deixaram de comprar recentemente

        .when(

            (F.col("score_r") <= 2)
            &
            (
                (F.col("score_f") >= 4)
                |
                (F.col("score_m") >= 4)
            ),

            "Grandes clientes em risco"
        )


        # Recência ruim e frequência intermediária

        .when(

            (F.col("score_r") <= 2)
            &
            (F.col("score_f") >= 3),

            "Em risco"
        )


        # Baixa recência e baixa frequência

        .when(

            (F.col("score_r") <= 2)
            &
            (F.col("score_f") <= 2),

            "Hibernando"
        )


        .otherwise(
            "Regulares"
        )
    )
)

In [0]:
df_sem_compra = (

    df_rfm_base

    .filter(
        F.col(
            "frequencia"
        ).isNull()
    )

    .withColumn(
        "score_r",
        F.lit(None).cast("int")
    )

    .withColumn(
        "score_f",
        F.lit(None).cast("int")
    )

    .withColumn(
        "score_m",
        F.lit(None).cast("int")
    )

    .withColumn(
        "codigo_rfm",
        F.lit(None).cast("string")
    )

    .withColumn(
        "score_rfm_medio",
        F.lit(None).cast("double")
    )

    .withColumn(
        "segmento_rfm",
        F.lit(
            "Sem compra"
        )
    )
)

In [0]:
df_clientes_rfm = (

    df_rfm_segmentado

    .unionByName(
        df_sem_compra,
        allowMissingColumns=True
    )
)

In [0]:
faturamento_total = (

    df_clientes_rfm

    .agg(
        F.sum(
            "valor_monetario"
        ).alias(
            "faturamento_total"
        )
    )

    .first()[
        "faturamento_total"
    ]
)


print(
    f"Faturamento total analisado: "
    f"R$ {faturamento_total:,.2f}"
)

In [0]:
df_clientes_rfm = (

    df_clientes_rfm

    .withColumn(

        "participacao_faturamento_percentual",

        F.when(
            F.col(
                "valor_monetario"
            ).isNotNull(),

            F.round(
                F.col(
                    "valor_monetario"
                )
                /
                F.lit(
                    faturamento_total
                )
                * 100,
                4
            )
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
df_clientes_rfm = (

    df_clientes_rfm

    .withColumn(

        "prioridade_acao",

        F.when(
            F.col(
                "segmento_rfm"
            )
            == "Grandes clientes em risco",

            "Prioridade muito alta"
        )

        .when(
            F.col(
                "segmento_rfm"
            )
            == "Em risco",

            "Prioridade alta"
        )

        .when(
            F.col(
                "segmento_rfm"
            )
            == "Potenciais leais",

            "Oportunidade"
        )

        .when(
            F.col(
                "segmento_rfm"
            )
            == "Novos clientes",

            "Desenvolvimento"
        )

        .when(
            F.col(
                "segmento_rfm"
            )
            == "Campeões",

            "Relacionamento"
        )

        .otherwise(
            "Monitoramento"
        )
    )
)

In [0]:
df_clientes_rfm = (

    df_clientes_rfm

    .select(

        "id_cliente",
        "nome_cliente",

        "segmento_cliente",
        "porte_cliente",
        "cidade",
        "estado",
        "situacao_cliente",

        "data_cadastro",

        "data_primeira_compra",
        "data_ultima_compra",

        "recencia_dias",
        "frequencia",
        "valor_monetario",
        "lucro_bruto",
        "ticket_medio",
        "tempo_relacionamento_dias",

        "score_r",
        "score_f",
        "score_m",

        "codigo_rfm",
        "score_rfm_medio",

        "segmento_rfm",

        "participacao_faturamento_percentual",

        "prioridade_acao",

        "_data_processamento"
    )
)

In [0]:
salvar_gold(
    df_clientes_rfm,
    "clientes_rfm"
)

In [0]:
display(

    df_clientes_rfm

    .groupBy(
        "segmento_rfm"
    )

    .agg(

        F.count("*").alias(
            "quantidade_clientes"
        ),

        F.round(
            F.sum(
                "valor_monetario"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.avg(
                "recencia_dias"
            ),
            1
        ).alias(
            "recencia_media"
        ),

        F.round(
            F.avg(
                "frequencia"
            ),
            1
        ).alias(
            "frequencia_media"
        )
    )

    .orderBy(
        F.desc(
            "faturamento"
        )
    )
)

In [0]:
df_resumo_segmentos_rfm = (

    df_clientes_rfm

    .groupBy(
        "segmento_rfm"
    )

    .agg(

        F.count(
            "*"
        ).alias(
            "quantidade_clientes"
        ),

        F.round(
            F.sum(
                "valor_monetario"
            ),
            2
        ).alias(
            "faturamento"
        ),

        F.round(
            F.sum(
                "lucro_bruto"
            ),
            2
        ).alias(
            "lucro_bruto"
        ),

        F.round(
            F.avg(
                "recencia_dias"
            ),
            2
        ).alias(
            "recencia_media_dias"
        ),

        F.round(
            F.avg(
                "frequencia"
            ),
            2
        ).alias(
            "frequencia_media"
        ),

        F.round(
            F.avg(
                "ticket_medio"
            ),
            2
        ).alias(
            "ticket_medio"
        ),

        F.round(
            F.avg(
                "valor_monetario"
            ),
            2
        ).alias(
            "valor_medio_cliente"
        )
    )
)

In [0]:
total_clientes = (
    df_clientes_rfm.count()
)


faturamento_total_rfm = (

    df_clientes_rfm

    .agg(
        F.sum(
            "valor_monetario"
        ).alias(
            "total"
        )
    )

    .first()[
        "total"
    ]
)

In [0]:
df_resumo_segmentos_rfm = (

    df_resumo_segmentos_rfm

    .withColumn(

        "participacao_clientes_percentual",

        F.round(
            F.col(
                "quantidade_clientes"
            )
            /
            F.lit(
                total_clientes
            )
            * 100,
            2
        )
    )

    .withColumn(

        "participacao_faturamento_percentual",

        F.round(
            F.col(
                "faturamento"
            )
            /
            F.lit(
                faturamento_total_rfm
            )
            * 100,
            2
        )
    )

    .withColumn(

        "margem_percentual",

        F.round(
            F.col(
                "lucro_bruto"
            )
            /
            F.col(
                "faturamento"
            )
            * 100,
            2
        )
    )

    .withColumn(
        "_data_processamento",
        F.current_timestamp()
    )
)

In [0]:
salvar_gold(
    df_resumo_segmentos_rfm,
    "resumo_segmentos_rfm"
)

In [0]:
display(

    df_resumo_segmentos_rfm

    .select(

        "segmento_rfm",

        "quantidade_clientes",

        "participacao_clientes_percentual",

        "faturamento",

        "participacao_faturamento_percentual",

        "recencia_media_dias",

        "frequencia_media",

        "ticket_medio",

        "margem_percentual"
    )

    .orderBy(
        F.desc(
            "faturamento"
        )
    )
)

In [0]:
display(

    df_clientes_rfm

    .filter(
        F.col(
            "segmento_rfm"
        ).isin(
            [
                "Grandes clientes em risco",
                "Em risco"
            ]
        )
    )

    .select(

        "id_cliente",
        "nome_cliente",
        "porte_cliente",

        "data_ultima_compra",

        "recencia_dias",
        "frequencia",
        "valor_monetario",

        "segmento_rfm",
        "prioridade_acao"
    )

    .orderBy(
        F.desc(
            "valor_monetario"
        )
    )

    .limit(30)
)

In [0]:
display(

    df_clientes_rfm

    .filter(
        F.col(
            "segmento_rfm"
        )
        == "Campeões"
    )

    .select(

        "id_cliente",
        "nome_cliente",
        "porte_cliente",

        "recencia_dias",
        "frequencia",
        "valor_monetario",
        "ticket_medio",

        "score_r",
        "score_f",
        "score_m"
    )

    .orderBy(
        F.desc(
            "valor_monetario"
        )
    )

    .limit(20)
)

In [0]:
faturamento_silver = (

    df_vendas

    .agg(
        F.round(
            F.sum(
                "valor_liquido"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()["total"]
)


faturamento_rfm = (

    df_clientes_rfm

    .agg(
        F.round(
            F.sum(
                "valor_monetario"
            ),
            2
        ).alias(
            "total"
        )
    )

    .first()["total"]
)


print(
    f"Silver: R$ {faturamento_silver:,.2f}"
)

print(
    f"RFM:    R$ {faturamento_rfm:,.2f}"
)

In [0]:
diferenca = abs(
    float(faturamento_silver)
    -
    float(faturamento_rfm)
)


if diferenca > 0.05:

    raise Exception(
        "O faturamento da análise RFM "
        "não corresponde ao faturamento da Silver."
    )


print(
    "Validação concluída: faturamento RFM consistente."
)

In [0]:
display(

    df_resumo_segmentos_rfm

    .select(
        "segmento_rfm",
        "participacao_clientes_percentual",
        "participacao_faturamento_percentual"
    )

    .orderBy(
        F.desc(
            "participacao_faturamento_percentual"
        )
    )
)

In [0]:
spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`clientes_rfm`
    IS 'Segmentação comportamental dos clientes utilizando Recência, Frequência e Valor Monetário.'
    """
)


spark.sql(
    f"""
    COMMENT ON TABLE
    `{catalogo_atual}`.`{schema_gold}`.`resumo_segmentos_rfm`
    IS 'Indicadores consolidados dos segmentos comportamentais definidos pela análise RFM.'
    """
)


print(
    "Descrições adicionadas às tabelas RFM."
)